# Qwen2-VL-7B → Edge AI (ASIC/CIM) 模型剪枝實驗

把 Qwen2-VL-7B Vision-Language Model 壓縮到 ASIC/CIM 可部署的大小。

**Pipeline:** Profile → 結構化剪枝 (ShortGPT BI Score) → INT4 量化

**需求:** GPU Runtime (T4 即可)

---

## 0. 環境設定

In [ ]:
!pip install -q torch torchvision transformers accelerate pillow

import torch
import json
from collections import defaultdict

# 確認 GPU
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠ 沒有 GPU！請到 Runtime → Change runtime type → T4 GPU")

In [ ]:
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"

## 1. Profile 模型架構

分析 Qwen2-VL 各 component 的參數量和記憶體佔用。

### 1a. Config-only (不下載權重)

In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Qwen2-VL 的 LM config 藏在 text_config 裡
lm_config = config.text_config if hasattr(config, 'text_config') else config

print("=" * 60)
print("MODEL CONFIGURATION — Qwen2-VL-7B")
print("=" * 60)

# Vision encoder
vc = config.vision_config
print(f"\n--- Vision Encoder (ViT) ---")
print(f"  Embed dim:       {vc.embed_dim}")
print(f"  Num layers:      {vc.depth}")
print(f"  Attention heads: {vc.num_heads}")
print(f"  MLP ratio:       {vc.mlp_ratio}")
print(f"  Patch size:      {vc.spatial_patch_size}")
print(f"  Spatial merge:   {vc.spatial_merge_size}×{vc.spatial_merge_size}")

# Language model
print(f"\n--- Language Model (Qwen2) ---")
print(f"  Hidden size:       {lm_config.hidden_size}")
print(f"  Intermediate size: {lm_config.intermediate_size}")
print(f"  Num layers:        {lm_config.num_hidden_layers}")
print(f"  Attention heads:   {lm_config.num_attention_heads}")
print(f"  KV heads:          {lm_config.num_key_value_heads} (GQA {lm_config.num_attention_heads}:{lm_config.num_key_value_heads})")
print(f"  Vocab size:        {lm_config.vocab_size}")

# Fusion
rope = lm_config.rope_scaling
print(f"\n--- Fusion ---")
print(f"  M-RoPE section:  {rope.get('mrope_section', 'N/A')} [temporal, height, width]")
print(f"  無 cross-attention（用 M-RoPE + PatchMerger fusion）")
print(f"  → 剪枝不需要保護 cross-attention 層，比 LLaMA Vision 簡單")

### 1b. Full Profiling (載入完整模型)

In [ ]:
from transformers import Qwen2VLForConditionalGeneration

print(f"Loading {MODEL_NAME}...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()
print("Model loaded!")

# 分析各 component
components = defaultdict(lambda: {"params": 0, "size_mb": 0, "layers": set()})

for name, param in model.named_parameters():
    n_params = param.numel()
    size_mb = param.nbytes / (1024 * 1024)

    if "visual" in name:
        cat = "patch_merger" if "merger" in name else "vision_encoder"
    elif "embed_tokens" in name or "lm_head" in name:
        cat = "embedding_head"
    else:
        cat = "language_model"

    components[cat]["params"] += n_params
    components[cat]["size_mb"] += size_mb
    parts = name.split(".")
    for i, p in enumerate(parts):
        if p in ("layers", "blocks") and i + 1 < len(parts) and parts[i + 1].isdigit():
            components[cat]["layers"].add(int(parts[i + 1]))

total_params = sum(c["params"] for c in components.values())
total_mb = sum(c["size_mb"] for c in components.values())

print(f"\n{'=' * 70}")
print(f"{'Component':<25} {'Params':>12} {'Size (MB)':>12} {'% Total':>10} {'Layers':>10}")
print("-" * 70)
for cat in ["vision_encoder", "patch_merger", "language_model", "embedding_head"]:
    if cat not in components:
        continue
    c = components[cat]
    pct = c["params"] / total_params * 100
    n_layers = len(c["layers"]) if c["layers"] else "-"
    print(f"{cat:<25} {c['params']:>12,} {c['size_mb']:>10.1f}MB {pct:>9.1f}% {str(n_layers):>10}")
print("-" * 70)
print(f"{'TOTAL':<25} {total_params:>12,} {total_mb:>10.1f}MB {100.0:>9.1f}%")

## 2. 結構化剪枝 — ShortGPT Block Influence Score

計算每個 LM layer 的 BI score，找出最冗餘的層。

BI = 1 - cosine_similarity(layer_input, layer_output)
- **BI 越低** = 該層改變越少 = 越可以被移除

In [ ]:
from transformers import AutoProcessor
from PIL import Image
import numpy as np
import torch.nn.functional as F

processor = AutoProcessor.from_pretrained(MODEL_NAME)

def get_lm_layers(model):
    """取得 LM layers。"""
    if hasattr(model, 'model') and hasattr(model.model, 'language_model'):
        return model.model.language_model.layers
    if hasattr(model, 'language_model'):
        return model.language_model.layers
    return model.model.layers

def prepare_calibration_data(processor, n_samples=16):
    """準備校準資料。"""
    samples = []
    text_prompts = [
        "Explain neural network pruning.",
        "What is the capital of France?",
        "Describe how transformers work.",
        "Write a Python sort function.",
        "What are benefits of quantization?",
        "How does attention mechanism work?",
        "What is knowledge distillation?",
        "Explain grouped query attention.",
    ]
    for prompt in text_prompts[:n_samples // 2]:
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], padding=True, return_tensors="pt")
        samples.append(inputs)

    image_prompts = [
        "Describe this image.",
        "What objects can you see?",
        "What is the main subject?",
        "Describe the colors.",
        "What patterns do you notice?",
        "Is there text in this image?",
        "Describe the layout.",
        "What is happening here?",
    ]
    for prompt in image_prompts[:n_samples // 2]:
        img = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
        messages = [{"role": "user", "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": prompt},
        ]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[img], padding=True, return_tensors="pt")
        samples.append(inputs)

    print(f"Prepared {len(samples)} calibration samples")
    return samples

samples = prepare_calibration_data(processor, n_samples=16)

In [ ]:
# 計算 Block Influence scores
layers = get_lm_layers(model)
n_layers = len(layers)
bi_scores = torch.zeros(n_layers)
n_valid = 0
device = next(model.parameters()).device

hidden_before = {}
hidden_after = {}
hooks = []

def make_pre_hook(idx):
    def hook(module, args):
        h = args[0] if isinstance(args[0], torch.Tensor) else args[0][0]
        hidden_before[idx] = h.detach().float().cpu()
    return hook

def make_post_hook(idx):
    def hook(module, args, output):
        h = output[0] if isinstance(output, tuple) else output
        hidden_after[idx] = h.detach().float().cpu()
    return hook

for i, layer in enumerate(layers):
    hooks.append(layer.register_forward_pre_hook(make_pre_hook(i)))
    hooks.append(layer.register_forward_hook(make_post_hook(i)))

print(f"Computing BI scores for {n_layers} layers...")
with torch.no_grad():
    for j, sample in enumerate(samples):
        try:
            sample = {k: v.to(device) if hasattr(v, 'to') else v for k, v in sample.items()}
            model(**sample)
            for i in range(n_layers):
                if i in hidden_before and i in hidden_after:
                    before = hidden_before[i].flatten()
                    after = hidden_after[i].flatten()
                    cos_sim = F.cosine_similarity(before.unsqueeze(0), after.unsqueeze(0))
                    bi_scores[i] += (1.0 - cos_sim.item())
            n_valid += 1
            print(f"  Sample {j+1}/{len(samples)} done")
        except Exception as e:
            print(f"  Sample {j+1} skipped: {e}")
        finally:
            hidden_before.clear()
            hidden_after.clear()

for h in hooks:
    h.remove()

if n_valid > 0:
    bi_scores /= n_valid

print(f"\nComputed from {n_valid} valid samples")

In [ ]:
# 顯示 BI scores
import matplotlib.pyplot as plt

print(f"{'=' * 60}")
print(f"BLOCK INFLUENCE SCORES — Qwen2-VL ({n_layers} LM layers)")
print(f"{'=' * 60}")
print(f"  lower BI = more redundant = safe to remove\n")

for i, score in enumerate(bi_scores):
    bar = '█' * int(max(0, score.item()) * 50)
    print(f"  Layer {i:3d}: {score.item():.4f} {bar}")

# 畫圖
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#ff4444' if s < 0.01 else '#58a6ff' if s < 0.1 else '#2ea043'
          for s in bi_scores.tolist()]
ax.bar(range(n_layers), bi_scores.tolist(), color=colors)
ax.set_xlabel('Layer Index')
ax.set_ylabel('Block Influence Score')
ax.set_title('Qwen2-VL-7B: Block Influence per LM Layer\n(Red = redundant, safe to prune)')
ax.set_xticks(range(n_layers))
plt.tight_layout()
plt.show()

# 排序找最冗餘的層
ranked = sorted(enumerate(bi_scores.tolist()), key=lambda x: x[1])
print(f"\n最冗餘的 6 層 (剪枝候選):")
for idx, score in ranked[:6]:
    print(f"  Layer {idx}: BI = {score:.6f}")

### 2b. 執行剪枝 (移除 20% LM layers)

In [ ]:
PRUNE_RATIO = 0.2  # 砍 20%

n_remove = int(n_layers * PRUNE_RATIO)
protected = {0, n_layers - 1}  # 保護首尾

candidates = [(i, bi_scores[i].item()) for i in range(n_layers) if i not in protected]
candidates.sort(key=lambda x: x[1])  # BI 最低的排前面

to_remove = sorted([idx for idx, _ in candidates[:n_remove]])
to_keep = [i for i in range(n_layers) if i not in to_remove]

print(f"剪枝計畫:")
print(f"  原始 LM layers:  {n_layers}")
print(f"  移除 layers:     {to_remove} ({len(to_remove)} layers)")
print(f"  保留 layers:     {to_keep} ({len(to_keep)} layers)")

In [ ]:
# 執行剪枝
layers = get_lm_layers(model)
new_layers = torch.nn.ModuleList([layers[i] for i in to_keep])

if hasattr(model, 'model') and hasattr(model.model, 'language_model'):
    model.model.language_model.layers = new_layers
elif hasattr(model, 'language_model'):
    model.language_model.layers = new_layers
else:
    model.model.layers = new_layers

# Qwen2-VL 的 num_hidden_layers 在 text_config 裡
if hasattr(model.config, 'text_config'):
    model.config.text_config.num_hidden_layers = len(new_layers)
else:
    model.config.num_hidden_layers = len(new_layers)

# 統計剪枝後
total_after = sum(p.numel() for p in model.parameters())
total_before = total_params  # from profiling step

print(f"\n剪枝完成!")
print(f"  Before: {total_before / 1e9:.2f}B params")
print(f"  After:  {total_after / 1e9:.2f}B params")
print(f"  Removed: {(total_before - total_after) / 1e9:.2f}B params ({(1 - total_after/total_before)*100:.1f}%)")
print(f"  LM layers: {n_layers} → {len(new_layers)}")

In [ ]:
# [可選] 儲存剪枝後的模型
# Colab Free RAM 有限，存模型容易 OOM
# 如果不需要存，直接跳到下一個 cell 就好

SAVE_MODEL = False  # 改成 True 如果你要存

if SAVE_MODEL:
    import os, gc
    PRUNED_PATH = "/content/pruned_model"
    os.makedirs(PRUNED_PATH, exist_ok=True)
    print(f"Saving to {PRUNED_PATH}...")
    torch.save(model.state_dict(), f"{PRUNED_PATH}/pytorch_model.bin")
    model.config.save_pretrained(PRUNED_PATH)
    processor.save_pretrained(PRUNED_PATH)
    print("Done!")
else:
    PRUNED_PATH = "(not saved)"
    print("跳過存模型（SAVE_MODEL=False）")
    print("重要結果已在上面的 BI scores + 剪枝計畫中")

## 3. 壓縮估算 + CIM Array Mapping

估算各壓縮方案的大小，以及對應的 CIM 陣列配置。

In [ ]:
import math

original_gb = 8.9e9 * 2 / (1024**3)
lm_kept = len(to_keep)

# 用 lm_config（在 cell-6 已定義，這裡再取一次以防重跑）
lm_cfg = config.text_config if hasattr(config, 'text_config') else config

scenarios = [
    ("Original BF16",            16, 1.0,  n_layers),
    ("INT8 only",                 8, 1.0,  n_layers),
    ("INT4 only",                 4, 1.0,  n_layers),
    (f"Prune {PRUNE_RATIO:.0%} + INT8", 8, 1-PRUNE_RATIO, lm_kept),
    (f"Prune {PRUNE_RATIO:.0%} + INT4", 4, 1-PRUNE_RATIO, lm_kept),
    (f"Prune {PRUNE_RATIO:.0%} + 2:4 + INT4", 4, 1-PRUNE_RATIO, lm_kept),
]

print(f"{'Scenario':<30} {'Bits':>5} {'Size':>7} {'Comp.':>6} {'LM Layers':>10}")
print("-" * 62)
for name, bits, keep, layers_kept in scenarios:
    size = original_gb * (bits / 16) * keep
    comp = original_gb / size if size > 0 else 0
    print(f"{name:<30} {bits:>3}bit {size:>5.1f}GB {comp:>5.1f}x {layers_kept:>10}")

# CIM array mapping
H = lm_cfg.hidden_size       # 3584
I = lm_cfg.intermediate_size # 18944
KV_H = lm_cfg.num_key_value_heads  # 4
N_H = lm_cfg.num_attention_heads   # 28
head_dim = H // N_H               # 128
K_dim = KV_H * head_dim           # 512
CIM = 256

def tiles(m, n, cim=CIM):
    return math.ceil(m/cim) * math.ceil(n/cim)

print(f"\n{'=' * 62}")
print(f"CIM ARRAY MAPPING (256×256)")
print(f"{'=' * 62}")
print(f"{'Matrix':<25} {'Shape':>15} {'Tiles':>8}")
print("-" * 50)
matrices = [
    ("Q_proj",    H, H,     tiles(H, H)),
    ("K_proj (GQA)", H, K_dim, tiles(H, K_dim)),
    ("V_proj (GQA)", H, K_dim, tiles(H, K_dim)),
    ("O_proj",    H, H,     tiles(H, H)),
    ("gate_proj", H, I,     tiles(H, I)),
    ("up_proj",   H, I,     tiles(H, I)),
    ("down_proj", I, H,     tiles(I, H)),
]
total_tiles_per_layer = 0
for name, m, n, t in matrices:
    print(f"  {name:<23} {m:>5}×{n:<5} {t:>8}")
    total_tiles_per_layer += t

print(f"\n  Total per layer: {total_tiles_per_layer:,} tiles")
print(f"  After pruning ({lm_kept} layers): {total_tiles_per_layer * lm_kept:,} tiles")
print(f"  Hidden {H} = {CIM}×{H//CIM} → 完美整除 CIM array")

## 4. 驗證剪枝後的模型

簡單測試剪枝後模型還能不能正常 inference。

In [ ]:
# 文字測試
test_prompt = "What is 2+2?"
messages = [{"role": "user", "content": [{"type": "text", "text": test_prompt}]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], padding=True, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=50)

response = processor.batch_decode(output_ids[:, inputs['input_ids'].shape[1]:],
                                   skip_special_tokens=True)[0]
print(f"Q: {test_prompt}")
print(f"A: {response}")
print(f"\n{'✓ 剪枝後模型可以正常 inference' if len(response) > 0 else '✗ 模型輸出異常'}")

In [ ]:
# 圖片測試
test_img = Image.fromarray(np.zeros((224, 224, 3), dtype=np.uint8))  # 黑色圖片
test_prompt = "Describe this image."
messages = [{"role": "user", "content": [
    {"type": "image", "image": test_img},
    {"type": "text", "text": test_prompt},
]}]
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[test_img], padding=True, return_tensors="pt").to(device)

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=100)

response = processor.batch_decode(output_ids[:, inputs['input_ids'].shape[1]:],
                                   skip_special_tokens=True)[0]
print(f"Q: {test_prompt}")
print(f"A: {response}")
print(f"\n{'✓ Vision 功能正常' if len(response) > 0 else '✗ Vision 輸出異常'}")

## 5. Summary

完成後的下一步：
1. **量化**: 用 AWQ 做 INT4 量化 (`pip install autoawq`)
2. **Export**: 輸出成 ONNX → ASIC compiler
3. **Benchmark**: 比較剪枝前後在 VQA/Caption 任務上的精度

In [ ]:
print(f"{'=' * 50}")
print(f"EXPERIMENT SUMMARY")
print(f"{'=' * 50}")
print(f"  Model:           {MODEL_NAME}")
print(f"  Prune ratio:     {PRUNE_RATIO:.0%}")
print(f"  LM layers:       {n_layers} → {lm_kept}")
print(f"  Params:          {total_before/1e9:.2f}B → {total_after/1e9:.2f}B")
print(f"  Removed layers:  {to_remove}")
print(f"  Saved to:        {PRUNED_PATH}")
print(f"\n  Next: quantize with AWQ INT4 → ~{original_gb*(4/16)*(1-PRUNE_RATIO):.1f}GB for ASIC/CIM")